# BE m228 Final Project

### Load Libraries & Data

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, f1_score
from sklearn.calibration import calibration_curve

from ucimlrepo import fetch_ucirepo 

In [18]:
diabetes_data = fetch_ucirepo(id=891)

X = diabetes_data.data.features 
X = X.drop('GenHlth', axis=1)
y = diabetes_data.data.targets

df = pd.DataFrame(X, columns=diabetes_data.variables['name'])
df['Diabetes_binary'] = y
df['ID'] = diabetes_data.data.ids
df.head(3)

name,ID,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,0,1,1,1,40,1,0,0,0,...,1,0,NaN,18,15,1,0,9,4,3
1,1,0,0,0,0,25,1,0,0,1,...,0,1,NaN,0,0,0,0,7,6,1
2,2,0,1,1,1,28,0,0,0,0,...,1,1,NaN,30,30,1,0,9,4,8


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(class_weight="balanced", random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "max_features": ["sqrt"]
}

# this optimization takes ~4 minutes to run, so commented out for now; best params are n_estimators=200, max_depth=10, max_features="sqrt"
# grid = GridSearchCV(rf, param_grid, cv=3, scoring="f1", n_jobs=-1)
# grid.fit(X_train, y_train)

# print(grid.best_params_)

C:\Users\schow\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 200}


In [21]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced", max_features="sqrt")
rf_model.fit(X_train, y_train)
rf_y_preds = rf_model.predict(X_test)

print(classification_report(y_test, rf_y_preds))

C:\Users\schow\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

           0       0.95      0.71      0.81     43667
           1       0.30      0.76      0.43      7069

    accuracy                           0.72     50736
   macro avg       0.62      0.73      0.62     50736
weighted avg       0.86      0.72      0.76     50736



In [22]:
rf_y_train_probs = rf_model.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, rf_y_train_probs)

# calculate F1 for every threshold
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Optimal Threshold found: {best_threshold:.4f}")

Optimal Threshold found: 0.6201


In [23]:
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]

# apply the custom threshold
rf_test_preds = (rf_test_probs >= best_threshold).astype(int)

#### Model Metrics

In [24]:
tn, fp, fn, tp = confusion_matrix(y_test, rf_y_preds).ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn) # sensitivity
specificity = tn / (tn + fp)
accuracy = (tp + tn) / (tp + fp + tn + fn)

print(f'precision of model: {precision}')
print(f'recall of model: {recall}')
print(f'specificity of model: {specificity}')
print(f'accuracy of model: {accuracy}')

precision of model: 0.2968585517203169
recall of model: 0.7579572782571792
specificity of model: 0.7093686307738108
accuracy of model: 0.7161384421318197


In [25]:
pr_auc = average_precision_score(y_test, rf_y_preds)
brier_score = brier_score_loss(y_test, rf_y_preds)
roc_auc = roc_auc_score(y_test, rf_y_preds)
f1 = f1_score(y_test, rf_y_preds)

print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Brier Score (Calibration): {brier_score:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

PR-AUC (Average Precision): 0.2587
Brier Score (Calibration): 0.2839
ROC-AUC: 0.7337
F1-Score: 0.4266


## XGBoost

Finding best parameters

In [26]:
from xgboost import XGBClassifier

ratio = (y_train.values == 0).sum() / (y_train.values == 1).sum()

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200]
}

xgb_model = XGBClassifier(
    scale_pos_weight=ratio, 
    eval_metric='logloss',
    random_state=42
)

# optimization commented out to save time; best params are max_depth=3, learning_rate=0.1, n_estimators=100
# grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='f1', cv=3, verbose=1)
# grid_search.fit(X_train, y_train)

# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Best F1-Score from GridSearch: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best F1-Score from GridSearch: 0.4293


Training, fitting, and predicting:

In [27]:
from sklearn.model_selection import StratifiedKFold, cross_validate

ratio = (y_train.values == 0).sum() / (y_train.values == 1).sum()

# 1 - initialize the Stratified K-Fold object
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2 - initialize the model 
# add scale_pos_weight to handle the imbalance during training
xg_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate = 0.1, scale_pos_weight=ratio, eval_metric='logloss', random_state=42)

# 3 - Run the Cross-Validation
# use 'f1' or 'roc_auc' because 'accuracy' is misleading for imbalanced data
cv_results = cross_validate(xg_model, X_train, y_train.values.ravel(), cv=skf, scoring=['roc_auc', 'f1', 'average_precision'])

In [32]:
xg_model.fit(X_train, y_train.values.ravel())

# find the best threshold using the training data
xg_y_train_probs = xg_model.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, xg_y_train_probs)

# calculate F1 for every threshold
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Optimal Threshold found: {best_threshold:.4f}")

Optimal Threshold found: 0.6357


In [33]:
xg_test_probs = xg_model.predict_proba(X_test)[:, 1]

# apply the custom threshold
xg_test_preds = (xg_test_probs >= best_threshold).astype(int)

#### Model Metrics

In [34]:
tn, fp, fn, tp = confusion_matrix(y_test, xg_test_preds).ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn) # sensitivity
specificity = tn / (tn + fp)
accuracy = (tp + tn) / (tp + fp + tn + fn)

print(f'precision of model: {precision}')
print(f'recall of model: {recall}')
print(f'specificity of model: {specificity}')
print(f'accuracy of model: {accuracy}')

precision of model: 0.36379370024229835
recall of model: 0.5947092941010044
specificity of model: 0.8316348730162365
accuracy of model: 0.7986242510249133


In [35]:
pr_auc = average_precision_score(y_test, xg_test_probs)
brier_score = brier_score_loss(y_test, xg_test_probs)
roc_auc = roc_auc_score(y_test, xg_test_probs)
f1 = f1_score(y_test, xg_test_preds)

print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Brier Score (Calibration): {brier_score:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

PR-AUC (Average Precision): 0.3991
Brier Score (Calibration): 0.1817
ROC-AUC: 0.8110
F1-Score: 0.4514


### Comparing the Models

Confusion Matrices

In [ ]:
xg_cm = confusion_matrix(y_test, xg_test_preds)
plt.figure(figsize=(6,4))
sns.heatmap(xg_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'Diabetic'], yticklabels=['Healthy', 'Diabetic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('XGBoost Confusion Matrix')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_y_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Healthy', 'Diabetic'], yticklabels=['Healthy', 'Diabetic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Random Forest Confusion Matrix')
plt.show()

Feature Importance

In [ ]:
# get importance based on 'Gain' (contribution to accuracy)
importance_gain = xg_model.get_booster().get_score(importance_type='gain')
xg_importances = pd.DataFrame.from_dict(importance_gain, orient='index', columns=['Gain'])

feature_importance_df = pd.DataFrame({
    'Feature': list(importance_gain.keys()),
    'Importance': list(importance_gain.values())
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(6, 4))
sns.barplot(data=feature_importance_df.head(10), x='Importance', y='Feature', palette='Blues_r')
plt.title('XGBoost Top 10 Features')
plt.xlabel('Importance Score (Gain)')
plt.ylabel('')
plt.show()

In [ ]:
rf_importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
rf_importances = rf_importances.sort_values('Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(6, 4))
sns.barplot(data=rf_importances.head(10), x='Importance', y='Feature', palette='Purples_r')
plt.title('Random Forest Top 10 Features')
plt.xlabel('Importance Score (Contribution)')
plt.ylabel('')
plt.show()

In [ ]:
# for row in rf_importances.itertuples():
    # print(f"{row.Feature}: {row.Importance:.6f}")

Precision-Recall Curves

In [ ]:
xg_precision, xg_recall, xg_thresholds = precision_recall_curve(y_test, xg_test_probs)
rf_precision, rf_recall, rf_thresholds = precision_recall_curve(y_test, rf_y_preds)
xg_pr_auc = auc(xg_recall, xg_precision)
rf_pr_auc = auc(rf_recall, rf_precision)

plt.figure(figsize=(6, 4))
plt.plot(rf_recall, rf_precision, color='lightsteelblue', lw=2, label=f'PR Curve (AUC = {rf_pr_auc:.2f})')
plt.plot(xg_recall, xg_precision, color='mediumpurple', lw=2, label=f'PR Curve (AUC = {xg_pr_auc:.2f})')

plt.fill_between(rf_recall, rf_precision, alpha=0.2, color='lightsteelblue')
plt.fill_between(xg_recall, xg_precision, alpha=0.2, color='mediumpurple')

# add a baseline for a random model (ratio of positive cases)
baseline_value = (y_test.sum() / len(y_test)).item()

plt.axhline(y=baseline_value, color='navy', linestyle='--', label=f'Baseline ({baseline_value:.2f})')

plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.show()

Calibration Curve

In [ ]:
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]
rf_prob_true, rf_prob_pred = calibration_curve(y_test, rf_test_probs, n_bins=10)

xg_test_probs = xg_model.predict_proba(X_test)[:, 1]
xg_prob_true, xg_prob_pred = calibration_curve(y_test, xg_test_probs, n_bins=10)

plt.figure(figsize=(6, 4))
plt.plot(rf_prob_pred, rf_prob_true, marker='s', label='Random Forest', color='mediumpurple')
plt.plot(xg_prob_pred, xg_prob_true, marker='o', label='XGBoost', color='darkblue')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated', color='black')

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives (Actual)')
plt.title('Calibration Curve (Reliability Diagram)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

mean absolute error? SHAP?